In [8]:
import pandas as pd
import numpy as np
from itertools import combinations
from typing import Dict, List, Any


# ==========================================================
# DATA PREPARATION
# ==========================================================
def clean_timeseries(
    df: pd.DataFrame,
    time_col: str,
    streams: List[str]
) -> pd.DataFrame:

    data = df.copy()
    data.columns = data.columns.str.strip()

    cols = [time_col] + streams
    data = data[cols]

    # Convert Unix timestamp to datetime
    data[time_col] = pd.to_datetime(
        data[time_col],
        unit="s",
        errors="coerce"
    )

    # Sort and set index
    data = data.sort_values(time_col)
    data = data.set_index(time_col)

    # Fill missing numeric values
    num_cols = data.select_dtypes(include=np.number).columns

    if len(num_cols) > 0:
        data[num_cols] = data[num_cols].interpolate(
            method="linear",
            limit_direction="both"
        )

    return data.dropna(how="all")


# ==========================================================
# WINDOW GENERATOR
# ==========================================================
def rolling_segments(
    df: pd.DataFrame,
    size: int,
    step: int
):

    total = len(df)

    for start in range(0, total - size + 1, step):
        end = start + size
        yield df.iloc[start:end]


# ==========================================================
# CORRELATION PER WINDOW
# ==========================================================
def correlation_by_window(
    df: pd.DataFrame,
    size: int,
    step: int,
    method: str = "pearson"
) -> List[Dict[str, Any]]:

    output = []

    for idx, chunk in enumerate(rolling_segments(df, size, step)):

        if len(chunk) < 2:
            continue

        matrix = chunk.corr(method=method)

        output.append({
            "window_index": idx,
            "start_time": chunk.index[0],
            "end_time": chunk.index[-1],
            "window_size": len(chunk),
            "correlation_matrix": matrix
        })

    return output


# ==========================================================
# WINDOW TO WINDOW CHANGES
# ==========================================================
def correlation_shift(
    results: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:

    shifts = []

    for prev, curr in zip(results[:-1], results[1:]):

        cols = sorted(
            set(prev["correlation_matrix"].columns) &
            set(curr["correlation_matrix"].columns)
        )

        for a, b in combinations(cols, 2):

            old_val = prev["correlation_matrix"].loc[a, b]
            new_val = curr["correlation_matrix"].loc[a, b]

            shifts.append({
                "window_index": curr["window_index"],
                "start_time": curr["start_time"],
                "end_time": curr["end_time"],
                "stream_1": a,
                "stream_2": b,
                "previous_corr": float(old_val),
                "current_corr": float(new_val),
                "delta": float(abs(new_val - old_val))
            })

    return shifts


# ==========================================================
# ALERTS
# ==========================================================
def build_alerts(
    changes: List[Dict[str, Any]],
    strong: float,
    weak: float,
    delta_limit: float
) -> List[Dict[str, Any]]:

    alerts = []

    for row in changes:

        if row["delta"] < delta_limit:
            continue

        prev = row["previous_corr"]
        curr = row["current_corr"]

        if prev >= strong and curr <= weak:
            level = "HIGH"
            msg = f"Strong correlation dropped sharply ({prev:.3f} → {curr:.3f})"

        elif prev * curr < 0:
            level = "MEDIUM"
            msg = f"Sign change detected ({prev:.3f} → {curr:.3f})"

        else:
            level = "LOW"
            msg = f"Significant change ({prev:.3f} → {curr:.3f})"

        item = row.copy()
        item["alert_level"] = level
        item["reason"] = msg

        alerts.append(item)

    return alerts


# ==========================================================
# MAIN PIPELINE
# ==========================================================
def run_correlation_monitor(
    df: pd.DataFrame,
    time_col: str,
    streams: List[str],
    window_size: int = 100,
    step_size: int = 20,
    method: str = "pearson",
    strong_corr: float = 0.7,
    weak_corr: float = 0.4,
    delta_limit: float = 0.25
) -> Dict[str, Any]:

    prepared = clean_timeseries(df, time_col, streams)

    matrices = correlation_by_window(
        prepared,
        size=window_size,
        step=step_size,
        method=method
    )

    changes = correlation_shift(matrices)

    alerts = build_alerts(
        changes,
        strong=strong_corr,
        weak=weak_corr,
        delta_limit=delta_limit
    )

    return {
        "processed_data": prepared,
        "correlation_results": matrices,
        "changes": changes,
        "alerts": alerts,
        "summary": {
            "total_windows": len(matrices),
            "total_changes": len(changes),
            "total_alerts": len(alerts),
            "window_size": window_size,
            "step_size": step_size
        }
    }


# ==========================================================
# EXECUTION
# ==========================================================
if __name__ == "__main__":

    df = pd.read_csv("most_correlated_streams_window.csv")

    result = run_correlation_monitor(
        df=df,
        time_col="time",
        streams=["s1", "s2", "s3"],
        window_size=100,
        step_size=20,
        delta_limit=0.25
    )

    # ======================================================
    # SUMMARY
    # ======================================================
    print("=== Correlation Change Detection Summary ===")
    print("Total windows :", result["summary"]["total_windows"])
    print("Total changes :", result["summary"]["total_changes"])
    print("Total alerts  :", result["summary"]["total_alerts"])
    print()

    # ======================================================
    # PRINT MATRICES
    # ======================================================
    print("=== Correlation Matrices for All Windows ===\n")

    for item in result["correlation_results"]:

        print(
            f"Window {item['window_index']:2d} | "
            f"From: {item['start_time']} → {item['end_time']} "
            f"(Size: {item['window_size']})"
        )

        print(item["correlation_matrix"].round(4))
        print("-" * 70)

    # ======================================================
    # PRINT ALERTS
    # ======================================================
    if result["alerts"]:

        alerts_df = pd.DataFrame(result["alerts"])

        print("\n=== Alerts Found ===")

        print(
            alerts_df[
                [
                    "window_index",
                    "start_time",
                    "stream_1",
                    "stream_2",
                    "previous_corr",
                    "current_corr",
                    "delta",
                    "alert_level"
                ]
            ].round(4)
        )

    else:
        print("\nNo significant correlation changes detected.")

=== Correlation Change Detection Summary ===
Total windows : 46
Total changes : 135
Total alerts  : 28

=== Correlation Matrices for All Windows ===

Window  0 | From: 1970-01-01 00:00:00 → 1970-01-01 00:01:39 (Size: 100)
        s1      s2      s3
s1  1.0000 -0.9534  1.0000
s2 -0.9534  1.0000 -0.9534
s3  1.0000 -0.9534  1.0000
----------------------------------------------------------------------
Window  1 | From: 1970-01-01 00:00:20 → 1970-01-01 00:01:59 (Size: 100)
        s1      s2      s3
s1  1.0000 -0.9657  1.0000
s2 -0.9657  1.0000 -0.9657
s3  1.0000 -0.9657  1.0000
----------------------------------------------------------------------
Window  2 | From: 1970-01-01 00:00:40 → 1970-01-01 00:02:19 (Size: 100)
        s1      s2      s3
s1  1.0000 -0.9652  1.0000
s2 -0.9652  1.0000 -0.9652
s3  1.0000 -0.9652  1.0000
----------------------------------------------------------------------
Window  3 | From: 1970-01-01 00:01:00 → 1970-01-01 00:02:39 (Size: 100)
        s1      s2      s